In [1]:
"""
compute_aqi.py
==============
Computes a CPCB-style Air Quality Index from the quality-checked daily
pollutant panel.

    python compute_aqi.py

INPUT   Bengaluru_AQ_Weather_daily_v3_noPBL.csv   (read-only)
OUTPUT  Bengaluru_AQI_daily.csv                   (input + AQI columns)
        aqi_summary.csv                           (distribution tables)

METHOD
------
CPCB National AQI (launched 17 Oct 2014). Each pollutant concentration is
converted to a sub-index by linear interpolation between health breakpoints,
and the WORST sub-index becomes the AQI for that station-day.

    sub-index = I_lo + (I_hi - I_lo) / (C_hi - C_lo) * (C - C_lo)

The breakpoint bands are contiguous in the formula even though CPCB prints them
as 31-60, 61-90 and so on. CPCB's own worked example confirms this: PM2.5 of
150 gives 300 + (150 - 120) * 100 / 130 = 323, using 120 rather than 121 as the
band's lower bound.

AVAILABILITY RULE
    At least 3 usable pollutants, and at least one of PM2.5 / PM10 must be
    among them. Days failing this get no AQI rather than a partial one.

=============================================================================
LIMITATION - READ BEFORE REPORTING THESE NUMBERS AS "CPCB AQI"
=============================================================================
CPCB specifies different averaging periods per pollutant:

    PM2.5, PM10, NO2, SO2   24-hour average      <- we have exactly this
    CO, Ozone               8-hour rolling MAX   <- we do NOT have this

Our source data is daily means only; the hourly readings were aggregated
upstream, so an 8-hour rolling maximum cannot be reconstructed. For CO and
Ozone this script uses the DAILY MEAN as a proxy.

A daily mean is always <= the 8-hour maximum of the same day, so both
sub-indices are systematically UNDERSTATED. The effect is largest for Ozone,
which has a strong afternoon peak that a daily mean flattens, and it matters
most in the pre-monsoon months when Ozone is highest.

Consequence: on any day where CO or Ozone would have been the worst pollutant,
this AQI may be too low and may name the wrong prominent pollutant. The output
column `aqi_proxy_dependent` marks exactly those rows.

These values should be described as a "CPCB-method AQI computed from daily
means" and NOT as an exact CPCB AQI.
=============================================================================
"""

import numpy as np
import pandas as pd

IN_CSV = "../data/processed/Bengaluru_AQ_Weather_daily_v3_noPBL.csv"
OUT_CSV = "../data/processed/Bengaluru_AQI_daily.csv"
OUT_SUMMARY = "../data/processed/aqi_summary.csv"

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

# Pollutants whose CPCB averaging basis we can honour exactly.
EXACT_BASIS = {"PM2.5", "PM10", "NO2", "SO2"}
# Pollutants where the daily mean stands in for an 8-hour rolling maximum.
PROXY_BASIS = {"CO", "Ozone"}

# Concentration cut points, contiguous, paired with index cut points.
# Units: µg/m³ except CO in mg/m³.
BREAKPOINTS = {
    "PM2.5": [0, 30, 60, 90, 120, 250],
    "PM10":  [0, 50, 100, 250, 350, 430],
    "NO2":   [0, 40, 80, 180, 280, 400],
    "SO2":   [0, 40, 80, 380, 800, 1600],
    "CO":    [0, 1.0, 2.0, 10, 17, 34],
    "Ozone": [0, 50, 100, 168, 208, 748],
}
INDEX_POINTS = [0, 50, 100, 200, 300, 400]

CATEGORIES = [
    (0, 50, "Good"),
    (50, 100, "Satisfactory"),
    (100, 200, "Moderate"),
    (200, 300, "Poor"),
    (300, 400, "Very Poor"),
    (400, 10_000, "Severe"),
]

AQI_MAX = 500          # CPCB scale ceiling
MIN_POLLUTANTS = 3


# ----------------------------------------------------------------------------
def sub_index(conc, pollutant):
    """Linear interpolation between the pollutant's health breakpoints.
    Above the top band, the slope of the final band is carried forward, then
    the result is capped at 500 (the top of the CPCB scale)."""
    if pd.isna(conc) or conc < 0:
        return np.nan

    bp = BREAKPOINTS[pollutant]
    for i in range(len(bp) - 1):
        c_lo, c_hi = bp[i], bp[i + 1]
        if conc <= c_hi:
            i_lo, i_hi = INDEX_POINTS[i], INDEX_POINTS[i + 1]
            return i_lo + (i_hi - i_lo) * (conc - c_lo) / (c_hi - c_lo)

    # Beyond the last breakpoint: extend the final band's slope.
    c_lo, c_hi = bp[-2], bp[-1]
    slope = (INDEX_POINTS[-1] - INDEX_POINTS[-2]) / (c_hi - c_lo)
    return min(INDEX_POINTS[-1] + (conc - c_hi) * slope, AQI_MAX)


def categorise(aqi):
    if pd.isna(aqi):
        return np.nan
    for lo, hi, name in CATEGORIES:
        if lo < aqi <= hi or (lo == 0 and aqi == 0):
            return name
    return np.nan


# ----------------------------------------------------------------------------
def compute(df):
    # Usable concentrations only. A sub-index built on a stuck sensor would
    # produce a plausible-looking AQI that is simply wrong.
    usable = {}
    for p in POLLUTANTS:
        qcol = f"{p}_quality"
        if qcol in df.columns:
            bad = df[qcol].astype(str).str.startswith("unusable")
        else:
            bad = pd.Series(False, index=df.index)
        usable[p] = df[p].where(~bad)

    # Sub-index per pollutant.
    sub = pd.DataFrame(index=df.index)
    for p in POLLUTANTS:
        sub[p] = usable[p].map(lambda c, pol=p: sub_index(c, pol))

    # Round once, here, and derive everything downstream from the rounded
    # values. Computing the AQI from unrounded sub-indices while publishing
    # rounded ones makes the AQI un-reproducible from the columns beside it:
    # 99.46 rounds to 99, but via 99.5 it rounds to 100.
    sub = sub.round(1)

    out = pd.DataFrame(index=df.index)
    for p in POLLUTANTS:
        out[f"subindex_{p}"] = sub[p]

    n_valid = sub.notna().sum(axis=1)
    has_pm = sub["PM2.5"].notna() | sub["PM10"].notna()
    computable = (n_valid >= MIN_POLLUTANTS) & has_pm

    aqi = sub.max(axis=1).where(computable)
    prominent = sub.idxmax(axis=1).where(computable)

    out["aqi"] = aqi.round(0)
    out["aqi_category"] = aqi.map(categorise)
    out["aqi_prominent_pollutant"] = prominent
    out["aqi_n_pollutants"] = n_valid.where(computable).astype("Int64")
    out["aqi_computable"] = computable

    # Which pollutants contributed, so a later reader can see whether a low
    # pollutant count is hiding the one that would have been worst.
    out["aqi_pollutants_used"] = sub.apply(
        lambda r: "|".join(sorted(p for p in POLLUTANTS if pd.notna(r[p]))),
        axis=1).where(computable)

    # The averaging-basis caveat, carried per row rather than left in a footnote.
    out["aqi_prominent_basis"] = prominent.map(
        lambda p: ("24h mean (CPCB basis)" if p in EXACT_BASIS
                   else "daily mean proxy for 8h max" if p in PROXY_BASIS
                   else np.nan))
    out["aqi_proxy_dependent"] = prominent.isin(PROXY_BASIS) & computable

    # A day where a proxy pollutant is CLOSE to the winner is also at risk of
    # being wrong, since the true 8-hour max could overtake the leader.
    proxy_max = sub[list(PROXY_BASIS)].max(axis=1)
    out["aqi_proxy_margin"] = (aqi - proxy_max).round(1).where(computable)

    return out


def validate(df, out):
    problems = []
    if len(out) != len(df):
        problems.append(f"row count changed: {len(df)} -> {len(out)}")

    ok = out["aqi_computable"]
    if out.loc[ok, "aqi"].isna().any():
        problems.append("computable row with no AQI value")
    if out.loc[~ok, "aqi"].notna().any():
        problems.append("non-computable row carries an AQI value")
    if (out["aqi"] < 0).any() or (out["aqi"] > AQI_MAX).any():
        problems.append(f"AQI outside 0-{AQI_MAX}")
    if out.loc[ok, "aqi_category"].isna().any():
        problems.append("computable row with no category")

    # The AQI must equal the worst sub-index present on that row.
    subs = out[[f"subindex_{p}" for p in POLLUTANTS]]
    recomputed = subs.max(axis=1).round(0)
    mismatch = ok & (recomputed - out["aqi"]).abs().gt(0.5)
    if mismatch.any():
        problems.append(f"{int(mismatch.sum())} rows where AQI != worst sub-index")

    if (out.loc[ok, "aqi_n_pollutants"] < MIN_POLLUTANTS).any():
        problems.append("computable row with fewer than 3 pollutants")

    pm_missing = ok & out["aqi_pollutants_used"].notna() & ~(
        out["aqi_pollutants_used"].str.contains("PM2.5")
        | out["aqi_pollutants_used"].str.contains("PM10"))
    if pm_missing.any():
        problems.append(f"{int(pm_missing.sum())} computable rows without PM2.5 or PM10")

    return problems


def self_test():
    """CPCB's published worked example, plus the band boundaries."""
    checks = [
        ("PM2.5", 150, 323),    # CPCB's own example: 300 + (150-120)*100/130
        ("PM2.5", 30, 50),      # band edge
        ("PM2.5", 60, 100),
        ("PM10", 100, 100),
        ("NO2", 40, 50),
        ("SO2", 80, 100),
        ("CO", 1.0, 50),
        ("CO", 2.0, 100),
        ("Ozone", 168, 200),
    ]
    bad = []
    for pol, conc, expected in checks:
        got = round(sub_index(conc, pol))
        if abs(got - expected) > 1:
            bad.append(f"{pol} {conc} -> {got}, expected {expected}")
    return bad


# ----------------------------------------------------------------------------
def main():
    bad = self_test()
    print("SUB-INDEX SELF-TEST:", "passed" if not bad else bad)
    if bad:
        raise SystemExit("breakpoint table is wrong - fix before proceeding")

    df = pd.read_csv(IN_CSV, parse_dates=["Date"])
    print(f"\nRead {IN_CSV}  {df.shape}")

    out = compute(df)
    problems = validate(df, out)
    print("\nVALIDATION:", "all checks passed" if not problems else "")
    for p in problems:
        print("  [FAIL]", p)
    if problems:
        raise SystemExit(1)

    merged = pd.concat([df, out], axis=1)
    merged.to_csv(OUT_CSV, index=False)
    print(f"\nWrote {OUT_CSV}  {merged.shape}")

    # ---- summary -----------------------------------------------------------
    ok = merged["aqi_computable"]
    print(f"\nAQI computable on {int(ok.sum()):,} of {len(merged):,} "
          f"station-days ({ok.mean()*100:.1f}%)")

    print("\nAQI distribution:")
    print(merged.loc[ok, "aqi"].describe().round(1).to_string())

    print("\nCategory counts:")
    order = [c[2] for c in CATEGORIES]
    cat = (merged.loc[ok, "aqi_category"].value_counts()
           .reindex(order).fillna(0).astype(int))
    cat_pct = (cat / cat.sum() * 100).round(1)
    print(pd.DataFrame({"days": cat, "%": cat_pct}).to_string())

    print("\nProminent pollutant (which one set the AQI):")
    prom = merged.loc[ok, "aqi_prominent_pollutant"].value_counts()
    print(pd.DataFrame({"days": prom,
                        "%": (prom / prom.sum() * 100).round(1)}).to_string())

    print("\nContributing pollutant count:")
    print(merged.loc[ok, "aqi_n_pollutants"].value_counts().sort_index().to_string())

    print("\nBy station:")
    by_st = merged[ok].groupby("Station").agg(
        days=("aqi", "size"), mean_aqi=("aqi", "mean"),
        max_aqi=("aqi", "max"),
        pct_moderate_plus=("aqi", lambda s: (s > 100).mean() * 100))
    print(by_st.round(1).to_string())

    print("\nSeasonal mean AQI:")
    seasons = {1: "Winter", 2: "Winter", 3: "Pre-monsoon", 4: "Pre-monsoon",
               5: "Pre-monsoon", 6: "Monsoon", 7: "Monsoon", 8: "Monsoon",
               9: "Monsoon", 10: "Post-monsoon", 11: "Post-monsoon",
               12: "Winter"}
    seas = merged[ok].assign(season=merged.loc[ok, "Date"].dt.month.map(seasons))
    print(seas.groupby("season")["aqi"].mean().round(1)
          .reindex(["Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"])
          .to_string())

    # ---- the averaging-basis caveat, quantified ----------------------------
    print("\n" + "=" * 70)
    print("PROXY EXPOSURE - how much the CO/Ozone limitation affects results")
    print("=" * 70)
    dep = merged.loc[ok, "aqi_proxy_dependent"]
    print(f"Days where CO or Ozone set the AQI: {int(dep.sum()):,} "
          f"({dep.mean()*100:.1f}%)")
    print("  On these days the AQI is likely UNDERSTATED, because the true")
    print("  8-hour maximum exceeds the daily mean we used.")

    near = merged.loc[ok & (merged["aqi_proxy_margin"] <= 25)
                      & ~merged["aqi_proxy_dependent"]]
    print(f"\nDays where a proxy pollutant was within 25 index points of the")
    print(f"winner: {len(near):,}. On these the true 8-hour maximum could have")
    print("overtaken the leader, so the prominent pollutant may be wrong.")

    summary = pd.DataFrame({"days": cat, "pct": cat_pct})
    summary.to_csv(OUT_SUMMARY)
    print(f"\nWrote {OUT_SUMMARY}")


if __name__ == "__main__":
    main()

SUB-INDEX SELF-TEST: passed

Read Bengaluru_AQ_Weather_daily_v3_noPBL.csv  (6576, 64)

VALIDATION: all checks passed

Wrote Bengaluru_AQI_daily.csv  (6576, 79)

AQI computable on 6,508 of 6,576 station-days (99.0%)

AQI distribution:
count    6508.0
mean       76.5
std        35.4
min        16.0
25%        50.0
50%        72.0
75%       100.0
max       351.0

Category counts:
              days     %
aqi_category            
Good          1656  25.4
Satisfactory  3244  49.8
Moderate      1576  24.2
Poor            26   0.4
Very Poor        6   0.1
Severe           0   0.0

Prominent pollutant (which one set the AQI):
                         days     %
aqi_prominent_pollutant            
PM10                     5186  79.7
PM2.5                     593   9.1
CO                        271   4.2
NO2                       243   3.7
Ozone                     188   2.9
SO2                        27   0.4

Contributing pollutant count:
aqi_n_pollutants
3      81
4     405
5    2420
6    360

In [2]:
"""
build_features.py
=================
Feature engineering for the Bengaluru AQI forecasting models.

    python build_features.py

INPUT   Bengaluru_AQI_daily.csv      (read-only)
OUTPUT  Bengaluru_features.csv       one row per station-day, features + target
        feature_dictionary.csv       every feature: group, window, leakage class

No models are fitted here.

=============================================================================
THE LEAKAGE RULE
=============================================================================
A feature on the row for day t may only use information that would genuinely
be in hand at the forecasting origin, i.e. the end of day t-1.

Every historical feature in this file is therefore built from a series that
has already been shifted by one day. A "3-day rolling mean" on row t is the
mean of days t-3, t-2, t-1 — it does not include day t.

Weather is the exception that needs care, and it is split into two groups
with different names so the distinction cannot be lost:

    wxobs_*   weather OBSERVED up to t-1. Available at forecast time. Safe.
    wxfc_*    weather ON day t. NOT available at forecast time from
              observations — at runtime this must come from a weather
              FORECAST, not a reanalysis.

`wxfc_*` columns are legitimate and usually improve accuracy, because in
production you really do have a weather forecast for tomorrow. But training on
reanalysis values and reporting the resulting accuracy overstates real
performance, since a reanalysis is more accurate than any forecast. If these
columns are used, the honest final evaluation re-runs with weather from
Open-Meteo's Historical Forecast API in their place.

Set INCLUDE_SAME_DAY_WEATHER = False to drop them entirely.

=============================================================================
THE MASKING RULE
=============================================================================
Cells flagged `unusable:*` by the repair pass are set to NaN before any
feature is computed. A rolling window that silently averaged over a
stuck-sensor stretch would inherit the fault invisibly.

Each window additionally requires a minimum share of real values
(MIN_WINDOW_FRAC). A 7-day mean built from 2 surviving days is not a 7-day
mean, and returning NaN is more honest than returning a number.
"""

import numpy as np
import pandas as pd

IN_CSV = "../data/processed/Bengaluru_AQI_daily.csv"
OUT_CSV = "../data/processed/Bengaluru_features.csv"
OUT_DICT = "../data/processed/feature_dictionary.csv"

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
INCLUDE_SAME_DAY_WEATHER = True

LAGS = [1, 2, 3, 7, 14]
ROLL_WINDOWS = [3, 7, 14, 30]
MIN_WINDOW_FRAC = 5 / 7          # share of a window that must be real values

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

# PM10 and PM2.5 set 89% of the AQI, and they are the only two that retained
# weather signal after deseasonalising. They get the full treatment; the other
# four get light context features only.
CORE = ["aqi", "PM2.5", "PM10"]
SECONDARY = ["NO2", "SO2", "CO", "Ozone"]

# Six weather variables, one per independent factor. The full set of 17 has 32
# pairs above rho 0.7, which leaves tree-model feature importances meaningless.
WEATHER = ["temp_mean", "temp_range", "dewpoint_mean",
           "rain_sum", "wind_speed_mean", "pressure_mean"]
WEATHER_ROLL = [3, 7]

TARGET = "aqi"


# ----------------------------------------------------------------------------
def min_periods(window):
    return max(2, int(np.ceil(window * MIN_WINDOW_FRAC)))


def masked_frame(df):
    """Usable values only. Unusable cells become NaN so no feature can inherit
    a known sensor fault."""
    out = df[["Station", "Date"]].copy()

    for p in POLLUTANTS:
        qcol = f"{p}_quality"
        bad = (df[qcol].astype(str).str.startswith("unusable")
               if qcol in df.columns else pd.Series(False, index=df.index))
        out[p] = df[p].where(~bad)

    # AQI is already NaN where it was not computable.
    out["aqi"] = df["aqi"]
    out["aqi_category"] = df["aqi_category"]
    out["aqi_prominent_pollutant"] = df["aqi_prominent_pollutant"]
    out["aqi_n_pollutants"] = df["aqi_n_pollutants"]

    for w in WEATHER + ["wind_dir_resultant"]:
        if w in df.columns:
            out[w] = df[w]
    return out


# ----------------------------------------------------------------------------
def build_station(g, feats, dictionary):
    """All features for one station. `g` is sorted by date with a complete
    daily index, so shift(1) is genuinely 'yesterday'."""
    g = g.sort_values("Date")

    def record(name, group, window, leakage, note):
        dictionary.append(dict(feature=name, group=group, window=window,
                               leakage=leakage, note=note))

    # ---- core series: lags, rolling mean/sd, change rates -------------------
    for col in CORE:
        if col not in g.columns:
            continue
        past = g[col].shift(1)          # everything downstream starts at t-1

        for L in LAGS:
            name = f"{col}_lag{L}"
            feats[name] = g[col].shift(L)
            record(name, "lag", f"t-{L}", "safe",
                   "observed value L days before the target day")

        for W in ROLL_WINDOWS:
            r = past.rolling(W, min_periods=min_periods(W))
            feats[f"{col}_rmean{W}"] = r.mean()
            feats[f"{col}_rstd{W}"] = r.std()
            record(f"{col}_rmean{W}", "rolling", f"t-{W} to t-1", "safe",
                   f"mean of the {W} days before the target day")
            record(f"{col}_rstd{W}", "rolling", f"t-{W} to t-1", "safe",
                   f"variability over the {W} days before the target day")

        # Momentum: how fast it was already moving at the forecast origin.
        feats[f"{col}_chg1"] = g[col].shift(1) - g[col].shift(2)
        feats[f"{col}_chg7"] = g[col].shift(1) - g[col].shift(8)
        record(f"{col}_chg1", "change", "t-2 to t-1", "safe",
               "day-over-day change as of the forecast origin")
        record(f"{col}_chg7", "change", "t-8 to t-1", "safe",
               "week-over-week change as of the forecast origin")

    # ---- secondary pollutants: light context only --------------------------
    for col in SECONDARY:
        if col not in g.columns:
            continue
        feats[f"{col}_lag1"] = g[col].shift(1)
        feats[f"{col}_rmean7"] = g[col].shift(1).rolling(
            7, min_periods=min_periods(7)).mean()
        record(f"{col}_lag1", "lag", "t-1", "safe",
               "yesterday's level; weather adds little for this pollutant")
        record(f"{col}_rmean7", "rolling", "t-7 to t-1", "safe",
               "recent level of a pollutant that rarely sets the AQI")

    # ---- weather observed up to t-1 ----------------------------------------
    for w in WEATHER:
        if w not in g.columns:
            continue
        feats[f"wxobs_{w}_lag1"] = g[w].shift(1)
        record(f"wxobs_{w}_lag1", "weather-observed", "t-1", "safe",
               "yesterday's weather; known at forecast time")
        for W in WEATHER_ROLL:
            name = f"wxobs_{w}_rmean{W}"
            feats[name] = g[w].shift(1).rolling(
                W, min_periods=min_periods(W)).mean()
            record(name, "weather-observed", f"t-{W} to t-1", "safe",
                   f"{W}-day weather average ending at the forecast origin")

    # Cumulative rain: wet deposition has memory beyond a single day.
    if "rain_sum" in g.columns:
        for W in [3, 7]:
            name = f"wxobs_rain_cum{W}"
            feats[name] = g["rain_sum"].shift(1).rolling(
                W, min_periods=1).sum()
            record(name, "weather-observed", f"t-{W} to t-1", "safe",
                   f"total rainfall over the {W} days before the target day")

    # ---- weather on the target day -----------------------------------------
    if INCLUDE_SAME_DAY_WEATHER:
        for w in WEATHER:
            if w in g.columns:
                feats[f"wxfc_{w}"] = g[w]
                record(f"wxfc_{w}", "weather-same-day", "t",
                       "requires-forecast",
                       "weather on the target day; in production this is a "
                       "FORECAST, so evaluating on reanalysis overstates "
                       "accuracy")

        # Wind direction as sin/cos. Averaging bearings arithmetically is
        # wrong (350 deg and 10 deg average to 180, the opposite way), so the
        # resultant direction is decomposed instead.
        if "wind_dir_resultant" in g.columns:
            rad = np.deg2rad(g["wind_dir_resultant"])
            feats["wxfc_wind_dir_sin"] = np.sin(rad)
            feats["wxfc_wind_dir_cos"] = np.cos(rad)
            for nm in ["wxfc_wind_dir_sin", "wxfc_wind_dir_cos"]:
                record(nm, "weather-same-day", "t", "requires-forecast",
                       "cyclical encoding of wind direction")

        # Interactions, particulates only. NO2, SO2 and CO showed no weather
        # relationship above |rho| 0.12 after deseasonalising, so interaction
        # terms there would add dimensionality without signal.
        for p in ["PM2.5", "PM10"]:
            base = g[p].shift(1)        # yesterday's level, known at origin
            for w, why in [("wind_speed_mean", "dilution"),
                           ("temp_range", "nocturnal trapping"),
                           ("rain_sum", "wet deposition")]:
                if w not in g.columns:
                    continue
                name = f"ix_{p}_x_{w}"
                feats[name] = base * g[w]
                record(name, "interaction", "t-1 level x t weather",
                       "requires-forecast",
                       f"{why}; particulates only, where weather has signal")

    # ---- calendar ----------------------------------------------------------
    doy = g["Date"].dt.dayofyear
    feats["cal_doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    feats["cal_doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    dow = g["Date"].dt.dayofweek
    feats["cal_dow_sin"] = np.sin(2 * np.pi * dow / 7)
    feats["cal_dow_cos"] = np.cos(2 * np.pi * dow / 7)
    feats["cal_is_weekend"] = (dow >= 5).astype(int)
    for nm, note in [("cal_doy_sin", "annual cycle"), ("cal_doy_cos", "annual cycle"),
                     ("cal_dow_sin", "weekly traffic cycle"),
                     ("cal_dow_cos", "weekly traffic cycle"),
                     ("cal_is_weekend", "weekly traffic cycle")]:
        record(nm, "calendar", "t", "safe",
               f"{note}; the date is known in advance, so this is not leakage")

    return feats


def add_city_features(df, feats, dictionary):
    """Cross-station context: what the rest of the city was doing yesterday.
    Leave-one-out, so a station is never fed its own value back."""
    for col in ["aqi", "PM10"]:
        wide = df.pivot(index="Date", columns="Station", values=col)
        loo = pd.DataFrame({s: wide.drop(columns=s).mean(axis=1)
                            for s in wide.columns})
        stacked = loo.stack().rename("v").reset_index()
        stacked.columns = ["Date", "Station", "v"]
        stacked = stacked.sort_values(["Station", "Date"])
        stacked["v"] = stacked.groupby("Station")["v"].shift(1)

        key = df[["Date", "Station"]].merge(
            stacked, on=["Date", "Station"], how="left")
        name = f"city_{col}_lag1"
        feats[name] = key["v"].values
        dictionary.append(dict(
            feature=name, group="cross-station", window="t-1", leakage="safe",
            note="mean of the OTHER five stations yesterday; leave-one-out so "
                 "a station never sees its own value"))
    return feats


# ----------------------------------------------------------------------------
def leakage_audit(df, feats, n_samples=40, seed=0):
    """The real test of the leakage rule.

    For a sample of (station, date) rows, rebuild the features using ONLY the
    data up to and including that date, then confirm every `safe` feature on
    that row is unchanged. If any feature peeked at a future row, truncating
    the future changes its value and the check fails.

    `requires-forecast` features are excluded by construction: they legitimately
    use day t, which truncation keeps.
    """
    rng = np.random.default_rng(seed)
    stations = df["Station"].unique()
    dates = df["Date"].sort_values().unique()
    # Avoid the first 40 days, where windows are still warming up.
    candidates = dates[40:]

    safe_cols = [c for c, lk in zip(feats.columns, feats.attrs["leakage"])
                 if lk == "safe"]

    failures, checked = [], 0
    for _ in range(n_samples):
        st = rng.choice(stations)
        dt = pd.Timestamp(rng.choice(candidates))

        truncated = df[(df["Station"] == st) & (df["Date"] <= dt)].copy()
        if len(truncated) < 40:
            continue

        sub_feats = pd.DataFrame(index=truncated.index)
        build_station(truncated, sub_feats, [])

        full_row = feats.loc[
            (df["Station"] == st) & (df["Date"] == dt), safe_cols]
        trunc_row = sub_feats.loc[
            truncated.index[-1:], [c for c in safe_cols if c in sub_feats]]
        if full_row.empty or trunc_row.empty:
            continue

        checked += 1
        for c in trunc_row.columns:
            a = full_row.iloc[0][c]
            b = trunc_row.iloc[0][c]
            if pd.isna(a) and pd.isna(b):
                continue
            if pd.isna(a) != pd.isna(b) or not np.isclose(
                    float(a), float(b), rtol=1e-9, atol=1e-9, equal_nan=True):
                failures.append((st, str(dt.date()), c, a, b))

    return checked, failures


# ----------------------------------------------------------------------------
def main():
    df = pd.read_csv(IN_CSV, parse_dates=["Date"])
    print(f"Read {IN_CSV}  {df.shape}")

    base = masked_frame(df).sort_values(["Station", "Date"]).reset_index(drop=True)

    dictionary = []
    feats = pd.DataFrame(index=base.index)
    parts = []
    for st, g in base.groupby("Station", sort=False):
        f = pd.DataFrame(index=g.index)
        build_station(g, f, dictionary if not parts else [])
        parts.append(f)
    feats = pd.concat(parts).sort_index()

    feats = add_city_features(base, feats, dictionary)

    fd = pd.DataFrame(dictionary).drop_duplicates(subset="feature")
    fd = fd.set_index("feature").reindex(feats.columns).reset_index()
    feats.attrs["leakage"] = fd["leakage"].fillna("safe").tolist()

    out = pd.concat([
        base[["Station", "Date"]],
        feats,
        base[[TARGET, "aqi_category", "aqi_n_pollutants"]],
    ], axis=1)

    # ---- validation --------------------------------------------------------
    print("\n" + "=" * 70)
    print("LEAKAGE AUDIT")
    print("=" * 70)
    print("Rebuilding features for sampled rows using only data up to that")
    print("date, then checking every 'safe' feature is unchanged.\n")
    checked, failures = leakage_audit(base, feats)
    print(f"  rows checked: {checked}")
    if failures:
        print(f"  [FAIL] {len(failures)} feature values changed when the")
        print("         future was truncated — those features look ahead:")
        for f in failures[:10]:
            print("   ", f)
        raise SystemExit(1)
    print("  [PASS] no 'safe' feature changed. None of them look ahead.")

    n_fc = int((fd["leakage"] == "requires-forecast").sum())
    print(f"\n  {n_fc} features are marked 'requires-forecast' — they use day-t")
    print("  weather, which is a FORECAST in production, not an observation.")
    print("  They are not leakage, but training and evaluating them on")
    print("  reanalysis overstates real-world accuracy.")

    print("\n" + "=" * 70)
    print("FEATURE SUMMARY")
    print("=" * 70)
    print(f"Total features: {len(feats.columns)}")
    print("\nBy group:")
    print(fd.groupby("group").size().to_string())
    print("\nBy leakage class:")
    print(fd.groupby("leakage").size().to_string())

    print("\nCoverage (rows with a usable value), worst 10 features:")
    cov = (feats.notna().mean() * 100).round(1).sort_values()
    print(cov.head(10).to_string())
    print("\n  Low coverage is expected for long windows at the start of each")
    print("  station's series, and wherever masked cells thin a window below")
    print(f"  the {MIN_WINDOW_FRAC:.0%} minimum.")

    usable_rows = out[TARGET].notna() & feats.notna().all(axis=1)
    print(f"\nRows with a target and every feature present: "
          f"{int(usable_rows.sum()):,} of {len(out):,}")
    print("  Tree models handle NaN natively, so rows with some missing")
    print("  features are still usable. This figure is the strict case.")

    out.to_csv(OUT_CSV, index=False)
    fd.to_csv(OUT_DICT, index=False)
    print(f"\nWrote {OUT_CSV}  {out.shape}")
    print(f"Wrote {OUT_DICT}  ({len(fd)} features documented)")

    print("\nBefore modelling: split chronologically, never randomly. A random")
    print("split puts tomorrow in training and yesterday in test, and every")
    print("lag feature then carries test-period information.")


if __name__ == "__main__":
    main()

Read Bengaluru_AQI_daily.csv  (6576, 79)

LEAKAGE AUDIT
Rebuilding features for sampled rows using only data up to that
date, then checking every 'safe' feature is unchanged.

  rows checked: 40
  [PASS] no 'safe' feature changed. None of them look ahead.

  14 features are marked 'requires-forecast' — they use day-t
  weather, which is a FORECAST in production, not an observation.
  They are not leakage, but training and evaluating them on
  reanalysis overstates real-world accuracy.

FEATURE SUMMARY
Total features: 94

By group:
group
calendar             5
change               6
cross-station        2
interaction          6
lag                 19
rolling             28
weather-observed    20
weather-same-day     8

By leakage class:
leakage
requires-forecast    14
safe                 80

Coverage (rows with a usable value), worst 10 features:
CO_rmean7        77.5
CO_lag1          77.8
NO2_rmean7       81.0
NO2_lag1         81.9
Ozone_rmean7     93.0
Ozone_lag1       93.6
PM2.5_rst

In [6]:
!pip install xgboost scikit-learn
!python train_forecast.py

  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 1.6/48.9 MB 44.4 MB/s eta 0:00:02
   ----- ---------------------------------- 7.3/48.9 MB 27.2 MB/s eta 0:00:02
   ----------- ---------------------------- 13.6/48.9 MB 27.9 MB/s eta 0:00:02
   ----------------- ---------------------- 21.2/48.9 MB 30.1 MB/s eta 0:00:01
   ----------------------- ---------------- 28.6/48.9 MB 31.4 MB/s eta 0:00:01
   ---------------------------- ----------- 34.6/48.9 MB 30.9 MB/s eta 0:00:01
   -------------------------------- ------- 39.8/48.9 MB 29.8 MB/s eta 0:00:01
   ------------------------------------- -- 45.4/48.9 MB 29.2 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 28.9 MB/s eta 0:00:01
   --------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
python: can't open file 'C:\\Users\\Akansha\\train_forecast.py': [Errno 2] No such file or directory


In [9]:
"""
train_forecast.py
=================
24-hour-ahead AQI forecasting: XGBoost quantile regression.

    pip install xgboost scikit-learn pandas numpy
    python train_forecast.py

INPUT   Bengaluru_features.csv
        feature_dictionary.csv
        Bengaluru_AQI_daily.csv      (for target provenance)
OUTPUT  forecast_predictions.csv
        forecast_metrics.csv
        feature_importance.csv
        lstm_feature_set.csv

WHAT IS BEING FORECAST
----------------------
Every feature on row t is built from information available at the end of day
t-1, except the `wxfc_*` group (tomorrow's weather, a forecast in production).
So predicting the AQI on row t from row t's features IS a 24-hour-ahead
forecast; no further shifting is applied.

THREE DECISIONS THAT SHAPE THE RESULT
-------------------------------------
1. CLEAN EVALUATION SUBSET.
   Roughly 6-8% of every pollutant series is estimated rather than measured,
   with individual runs up to 62 days. An estimated series is locally linear,
   so "tomorrow equals today" predicts it almost perfectly and no model can
   beat that. Scoring on those days measures the interpolation, not the
   forecast.

   A test day is CLEAN when the pollutant that set its AQI was measured, not
   estimated. The AQI equals that pollutant's sub-index, so if the value was
   estimated, so is the AQI.

   Imputed days are KEPT IN TRAINING for this run — the aim is to isolate the
   measurement problem from the modelling one. If the model later proves to be
   learning interpolation artefacts, exclude them from training too and compare.

2. POOLED MODEL AS PRIMARY.
   Each station has only ~876 training rows against ~90 features. A pooled
   model sees six times the data and learns the shared physics once, with
   station identity passed as a feature so it can still separate them. The
   per-station models are retained for comparison, not as the headline.

3. CO AND NO2 FEATURES DROPPED.
   Not for the LSTM's reason (NaN intolerance) but for a worse one: CO is
   unusable from Jan 2026 and the test split starts in late 2025, so these
   features are largely present in training and absent in testing. A model
   that learned splits on them is handed NaN exactly where it is scored.
   That train/test availability shift degrades the period being measured.

PERSISTENCE BASELINE
--------------------
"Tomorrow equals today" is strong for daily AQI because the series is highly
autocorrelated. A model that cannot beat it has not earned its complexity, and
a MAPE reported without it is uninformative.
"""

import numpy as np
import pandas as pd
import xgboost as xgb

IN_FEATURES = "../data/processed/Bengaluru_features.csv"
IN_DICT = "../data/processed/feature_dictionary.csv"
IN_AQI = "../data/processed/Bengaluru_AQI_daily.csv"

OUT_PRED = "../data/processed/forecast_predictions.csv"
OUT_METRICS = "../data/processed/forecast_metrics.csv"
OUT_IMPORTANCE = "../data/processed/feature_importance.csv"
OUT_LSTM_FEATURES = "../data/processed/lstm_feature_set.csv"

TARGET = "aqi"
QUANTILES = [0.10, 0.50, 0.90]
TEST_FRACTION = 0.20
EXCEEDANCE_THRESHOLD = 200

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

# Dropped from BOTH models, for different reasons (see header).
EXCLUDE_PREFIXES = ("CO_", "NO2_")

EXCLUDE_IMPUTED_FROM_TRAINING = False     # see decision 1

XGB_PARAMS = dict(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=2.0,
    objective="reg:quantileerror",
    tree_method="hist",
    random_state=42,
)


# ----------------------------------------------------------------------------
def pinball_loss(y, pred, alpha):
    d = np.asarray(y) - np.asarray(pred)
    return float(np.mean(np.maximum(alpha * d, (alpha - 1) * d)))


def metrics_block(y, q10, q50, q90, persistence, label):
    y = np.asarray(y, dtype=float)
    q10, q50, q90 = map(np.asarray, (q10, q50, q90))
    err = q50 - y
    nz = y > 0

    out = {
        "scope": label,
        "n_days": int(len(y)),
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "mape_pct": float(np.mean(np.abs(err[nz] / y[nz])) * 100) if nz.any() else np.nan,
        "bias": float(np.mean(err)),
        "pinball_q10": pinball_loss(y, q10, 0.10),
        "pinball_q50": pinball_loss(y, q50, 0.50),
        "pinball_q90": pinball_loss(y, q90, 0.90),
        "interval_coverage_pct": float(np.mean((y >= q10) & (y <= q90)) * 100),
        "interval_width_mean": float(np.mean(q90 - q10)),
    }
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    out["r2"] = 1 - float(np.sum(err ** 2)) / ss_tot if ss_tot > 0 else np.nan

    p = np.asarray(persistence, dtype=float)
    ok = ~np.isnan(p)
    if ok.any():
        out["baseline_mae"] = float(np.mean(np.abs(p[ok] - y[ok])))
        out["skill_vs_persistence_pct"] = (
            (out["baseline_mae"] - out["mae"]) / out["baseline_mae"] * 100)
    return out


def exceedance_block(y, pred, threshold, tag):
    actual = np.asarray(y) > threshold
    flagged = np.asarray(pred) > threshold
    tp = int(np.sum(actual & flagged))
    fp = int(np.sum(~actual & flagged))
    fn = int(np.sum(actual & ~flagged))
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    return {f"exceed_{tag}_actual": int(actual.sum()),
            f"exceed_{tag}_tp": tp, f"exceed_{tag}_fp": fp,
            f"exceed_{tag}_fn": fn,
            f"exceed_{tag}_recall": recall,
            f"exceed_{tag}_precision": precision}


# ----------------------------------------------------------------------------
def target_provenance(path):
    """A day is CLEAN when the pollutant that set its AQI was measured.

    Also computed, more strictly: every contributing pollutant measured. The
    strict version is reported for context but not used to filter, since the
    AQI value itself is determined solely by the prominent pollutant.
    """
    aqi = pd.read_csv(path, parse_dates=["Date"])

    prom = aqi["aqi_prominent_pollutant"]
    clean = pd.Series(False, index=aqi.index)
    for p in POLLUTANTS:
        col = f"{p}_is_imputed"
        if col not in aqi.columns:
            continue
        is_p = prom == p
        clean |= is_p & ~aqi[col].astype("boolean").fillna(False).to_numpy()

    strict = pd.Series(True, index=aqi.index)
    for p in POLLUTANTS:
        col, q = f"{p}_is_imputed", f"{p}_quality"
        if col not in aqi.columns:
            continue
        contributes = ~aqi[q].astype(str).str.startswith("unusable")
        strict &= ~(contributes & aqi[col].astype("boolean").fillna(False).to_numpy())

    return pd.DataFrame({
        "Station": aqi["Station"], "Date": aqi["Date"],
        "target_clean": clean.to_numpy(),
        "target_clean_strict": (clean & strict).to_numpy(),
    })


def split_chronologically(g):
    g = g.sort_values("Date")
    cut = int(len(g) * (1 - TEST_FRACTION))
    return g.iloc[:cut], g.iloc[cut:]


def fit_quantiles(X_tr, y_tr, X_te):
    """Three models, then sort each row so the quantiles cannot cross."""
    models, preds = {}, []
    for a in QUANTILES:
        m = xgb.XGBRegressor(**dict(XGB_PARAMS, quantile_alpha=a))
        m.fit(X_tr, y_tr, verbose=False)
        models[a] = m
        preds.append(m.predict(X_te))
    s = np.sort(np.vstack(preds), axis=0)
    return models, s[0], s[1], s[2]


# ----------------------------------------------------------------------------
def main():
    df = pd.read_csv(IN_FEATURES, parse_dates=["Date"])
    fd = pd.read_csv(IN_DICT)
    prov = target_provenance(IN_AQI)
    print(f"Read {IN_FEATURES}  {df.shape}")

    df = df.merge(prov, on=["Station", "Date"], how="left", validate="one_to_one")
    df["target_clean"] = df["target_clean"].fillna(False)

    meta = ["Station", "Date", TARGET, "aqi_category", "aqi_n_pollutants",
            "target_clean", "target_clean_strict"]
    all_features = [c for c in df.columns if c not in meta]
    features = [c for c in all_features if not c.startswith(EXCLUDE_PREFIXES)]
    dropped = [c for c in all_features if c.startswith(EXCLUDE_PREFIXES)]

    key = "feature" if "feature" in fd.columns else fd.columns[0]
    fc_features = [f for f in fd.loc[fd["leakage"] == "requires-forecast", key]
                   if f in features]

    print(f"{len(all_features)} features available; {len(dropped)} dropped "
          f"(CO/NO2 availability shift): {dropped}")
    print(f"{len(features)} used, of which {len(fc_features)} forecast-dependent")

    df = df[df[TARGET].notna()].copy()
    print(f"{len(df):,} rows carry an AQI target")

    # ---- split -------------------------------------------------------------
    tr_parts, te_parts = [], []
    for _, g in df.groupby("Station", sort=True):
        a, b = split_chronologically(g)
        tr_parts.append(a)
        te_parts.append(b)
    train, test = pd.concat(tr_parts), pd.concat(te_parts)

    if EXCLUDE_IMPUTED_FROM_TRAINING:
        train = train[train["target_clean"]]
        print(f"Training rows after excluding imputed targets: {len(train):,}")

    clean = test["target_clean"].to_numpy()
    print(f"\nTest days: {len(test):,} total, {int(clean.sum()):,} clean "
          f"({clean.mean()*100:.1f}%), {int((~clean).sum()):,} excluded as "
          f"imputed-target")
    print(f"  stricter definition (every contributing pollutant measured): "
          f"{int(test['target_clean_strict'].sum()):,}")

    print("\n  Clean test days by station:")
    bs = test.groupby("Station")["target_clean"].agg(["sum", "size"])
    bs["pct"] = (bs["sum"] / bs["size"] * 100).round(1)
    print(bs.rename(columns={"sum": "clean", "size": "total"}).to_string())

    # ---- pooled model (primary) --------------------------------------------
    codes = {s: i for i, s in enumerate(sorted(df["Station"].unique()))}
    Xtr = train[features].copy()
    Xte = test[features].copy()
    Xtr["station_id"] = train["Station"].map(codes).values
    Xte["station_id"] = test["Station"].map(codes).values

    pooled_models, p10, p50, p90 = fit_quantiles(Xtr, train[TARGET], Xte)

    # ---- per-station models (comparison) -----------------------------------
    ps_rows = []
    for st, g_te in test.groupby("Station", sort=True):
        g_tr = train[train["Station"] == st]
        _, s10, s50, s90 = fit_quantiles(
            g_tr[features], g_tr[TARGET], g_te[features])
        ps_rows.append(pd.DataFrame({
            "Station": st, "Date": g_te["Date"].values,
            "ps_q10": s10, "ps_q50": s50, "ps_q90": s90}))
    ps = pd.concat(ps_rows, ignore_index=True)

    pred = pd.DataFrame({
        "Station": test["Station"].values,
        "Date": test["Date"].values,
        "actual": test[TARGET].values,
        "q10": p10, "q50": p50, "q90": p90,
        "persistence": test["aqi_lag1"].values,
        "target_clean": clean,
    }).merge(ps, on=["Station", "Date"], how="left")

    # ---- metrics -----------------------------------------------------------
    rows = []
    for label, mask in [("pooled ALL test days", np.ones(len(pred), bool)),
                        ("pooled CLEAN only", pred["target_clean"].to_numpy())]:
        sub = pred[mask]
        m = metrics_block(sub["actual"], sub["q10"], sub["q50"], sub["q90"],
                          sub["persistence"], label)
        m.update(exceedance_block(sub["actual"], sub["q90"],
                                  EXCEEDANCE_THRESHOLD, "q90"))
        m.update(exceedance_block(sub["actual"], sub["q50"],
                                  EXCEEDANCE_THRESHOLD, "q50"))
        rows.append(m)

    for label, mask in [("per-station ALL test days", np.ones(len(pred), bool)),
                        ("per-station CLEAN only", pred["target_clean"].to_numpy())]:
        sub = pred[mask]
        rows.append(metrics_block(sub["actual"], sub["ps_q10"], sub["ps_q50"],
                                  sub["ps_q90"], sub["persistence"], label))

    for st, sub in pred[pred["target_clean"]].groupby("Station", sort=True):
        rows.append(metrics_block(sub["actual"], sub["q10"], sub["q50"],
                                  sub["q90"], sub["persistence"],
                                  f"pooled CLEAN — {st}"))

    metrics = pd.DataFrame(rows)

    # ---- report ------------------------------------------------------------
    def show(m):
        print(f"  MAE {m['mae']:7.2f}   persistence {m.get('baseline_mae', float('nan')):7.2f}"
              f"   skill {m.get('skill_vs_persistence_pct', float('nan')):+7.1f}%"
              f"   MAPE {m['mape_pct']:5.1f}%   R2 {m['r2']:.3f}   n={m['n_days']:,}")

    print("\n" + "=" * 74)
    print("THE COMPARISON THAT MATTERS")
    print("=" * 74)
    print("\npooled, all test days (includes interpolated targets):")
    show(rows[0])
    print("\npooled, CLEAN test days only:")
    show(rows[1])
    print("\nper-station, all test days:")
    show(rows[2])
    print("\nper-station, CLEAN test days only:")
    show(rows[3])

    cp, ca = rows[1], rows[0]
    print("\n  Interpretation:")
    if cp.get("skill_vs_persistence_pct", -1) > 0:
        print("  On measured days the model beats persistence. The earlier")
        print("  negative skill came from interpolated targets, which are")
        print("  locally linear and therefore trivially persistent.")
    else:
        print("  The model still loses to persistence on measured days, so")
        print("  interpolation was not the whole story. Next levers: fewer")
        print("  features, stronger regularisation, or a longer test window.")
    print(f"  Skill moved from {ca.get('skill_vs_persistence_pct', float('nan')):+.1f}% "
          f"(all days) to {cp.get('skill_vs_persistence_pct', float('nan')):+.1f}% "
          f"(clean days).")

    print("\n" + "=" * 74)
    print("PER-STATION, POOLED MODEL, CLEAN DAYS")
    print("=" * 74)
    for m in rows[4:]:
        print(f"  {m['scope'].split('— ')[-1]:22s} "
              f"MAE {m['mae']:6.2f}  persistence "
              f"{m.get('baseline_mae', float('nan')):6.2f}  skill "
              f"{m.get('skill_vs_persistence_pct', float('nan')):+6.1f}%  "
              f"n={m['n_days']}")

    print("\n" + "=" * 74)
    print(f"EXCEEDANCE (AQI > {EXCEEDANCE_THRESHOLD}), clean days")
    print("=" * 74)
    n_act = cp["exceed_q90_actual"]
    print(f"  actual exceedance days: {n_act}")
    if n_act == 0:
        print("\n  None in the clean test set. Recall and precision are")
        print("  undefined and the alert component CANNOT be validated here.")
        print("  That is a fact about Bengaluru, not a defect: only ~32 of")
        print("  6,508 station-days in the whole record exceed 200. Either")
        print("  lower the alert threshold to a level that occurs, or state")
        print("  plainly that the surge model is unvalidated at this end.")
    else:
        print(f"  90th percentile: caught {cp['exceed_q90_tp']}, missed "
              f"{cp['exceed_q90_fn']}, false alarms {cp['exceed_q90_fp']}")
        print(f"    recall {cp['exceed_q90_recall']:.2f}  "
              f"precision {cp['exceed_q90_precision']:.2f}")
        print("  Recall matters more than precision for a health alert: a")
        print("  false alarm costs preparation, a miss costs patients.")

    print(f"\n  80% interval coverage (clean): {cp['interval_coverage_pct']:.1f}% "
          f"(target 80%), mean width {cp['interval_width_mean']:.1f}")
    if cp["interval_coverage_pct"] < 70:
        print("  Intervals are too narrow — the 90th percentile understates")
        print("  risk, which is why exceedance days go uncaught.")

    # ---- importance --------------------------------------------------------
    imp = pd.Series(pooled_models[0.50].feature_importances_,
                    index=list(Xtr.columns)).sort_values(ascending=False)
    imp.to_csv(OUT_IMPORTANCE, header=["importance"])
    print("\n" + "=" * 74)
    print("FEATURE IMPORTANCE (pooled median model)")
    print("=" * 74)
    print(imp.head(15).round(4).to_string())
    fc_share = float(imp[imp.index.isin(fc_features)].sum())
    print(f"\n  Forecast-dependent share: {fc_share*100:.1f}% — the part of")
    print("  performance that depends on having tomorrow's weather, and so")
    print("  the size of the gap the Historical Forecast re-run must close.")
    print("  Importances split between correlated features; read groups.")

    lstm_features = [f for f in features]
    pd.DataFrame({"feature": lstm_features}).to_csv(OUT_LSTM_FEATURES, index=False)

    pred.to_csv(OUT_PRED, index=False)
    metrics.to_csv(OUT_METRICS, index=False)
    print(f"\nWrote {OUT_PRED}  {pred.shape}")
    print(f"Wrote {OUT_METRICS}, {OUT_IMPORTANCE}, {OUT_LSTM_FEATURES}")
    print(f"  LSTM feature set: {len(lstm_features)} features "
          f"(CO/NO2 already excluded)")


if __name__ == "__main__":
    main()

Read Bengaluru_features.csv  (6576, 99)
94 features available; 4 dropped (CO/NO2 availability shift): ['NO2_lag1', 'NO2_rmean7', 'CO_lag1', 'CO_rmean7']
90 used, of which 14 forecast-dependent
6,508 rows carry an AQI target

Test days: 1,306 total, 1,180 clean (90.4%), 126 excluded as imputed-target
  stricter definition (every contributing pollutant measured): 1,007

  Clean test days by station:
                     clean  total   pct
Station                                
BTM Layout             203    220  92.3
Hebbal                 195    208  93.8
Jayanagar 5th Block    219    220  99.5
Kasturi Nagar          174    218  79.8
Peenya                 174    220  79.1
Silk Board             215    220  97.7

THE COMPARISON THAT MATTERS

pooled, all test days (includes interpolated targets):
  MAE   10.23   persistence   10.17   skill    -0.7%   MAPE  11.3%   R2 0.715   n=1,306

pooled, CLEAN test days only:
  MAE   10.85   persistence   11.01   skill    +1.5%   MAPE  11.7%   R2 0.6

In [10]:
"""
build_features.py
=================
Feature engineering for the Bengaluru AQI forecasting models.

    python build_features.py

INPUT   Bengaluru_AQI_daily.csv      (read-only)
OUTPUT  Bengaluru_features.csv       one row per station-day, features + target
        feature_dictionary.csv       every feature: group, window, leakage class

No models are fitted here.

=============================================================================
THE LEAKAGE RULE
=============================================================================
A feature on the row for day t may only use information that would genuinely
be in hand at the forecasting origin, i.e. the end of day t-1.

Every historical feature in this file is therefore built from a series that
has already been shifted by one day. A "3-day rolling mean" on row t is the
mean of days t-3, t-2, t-1 — it does not include day t.

Weather is the exception that needs care, and it is split into two groups
with different names so the distinction cannot be lost:

    wxobs_*   weather OBSERVED up to t-1. Available at forecast time. Safe.
    wxfc_*    weather ON day t. NOT available at forecast time from
              observations — at runtime this must come from a weather
              FORECAST, not a reanalysis.

`wxfc_*` columns are legitimate and usually improve accuracy, because in
production you really do have a weather forecast for tomorrow. But training on
reanalysis values and reporting the resulting accuracy overstates real
performance, since a reanalysis is more accurate than any forecast. If these
columns are used, the honest final evaluation re-runs with weather from
Open-Meteo's Historical Forecast API in their place.

Set INCLUDE_SAME_DAY_WEATHER = False to drop them entirely.

=============================================================================
THE MASKING RULE
=============================================================================
Cells flagged `unusable:*` by the repair pass are set to NaN before any
feature is computed. A rolling window that silently averaged over a
stuck-sensor stretch would inherit the fault invisibly.

Each window additionally requires a minimum share of real values
(MIN_WINDOW_FRAC). A 7-day mean built from 2 surviving days is not a 7-day
mean, and returning NaN is more honest than returning a number.
"""

import numpy as np
import pandas as pd

IN_CSV = "../data/processed/Bengaluru_AQI_daily.csv"
OUT_CSV = "../data/processed/Bengaluru_features.csv"
OUT_DICT = "../data/processed/feature_dictionary.csv"

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
INCLUDE_SAME_DAY_WEATHER = True

LAGS = [1, 2, 3, 7, 14]
ROLL_WINDOWS = [3, 7, 14, 30]
MIN_WINDOW_FRAC = 5 / 7          # share of a window that must be real values

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

# PM10 and PM2.5 set 89% of the AQI, and they are the only two that retained
# weather signal after deseasonalising. They get the full treatment; the other
# four get light context features only.
CORE = ["aqi", "PM2.5", "PM10"]
SECONDARY = ["NO2", "SO2", "CO", "Ozone"]

# Six weather variables, one per independent factor. The full set of 17 has 32
# pairs above rho 0.7, which leaves tree-model feature importances meaningless.
WEATHER = ["temp_mean", "temp_range", "dewpoint_mean",
           "rain_sum", "wind_speed_mean", "pressure_mean"]
WEATHER_ROLL = [3, 7]

TARGET = "aqi"


# ----------------------------------------------------------------------------
def min_periods(window):
    return max(2, int(np.ceil(window * MIN_WINDOW_FRAC)))


def masked_frame(df):
    """Usable values only. Unusable cells become NaN so no feature can inherit
    a known sensor fault."""
    out = df[["Station", "Date"]].copy()

    for p in POLLUTANTS:
        qcol = f"{p}_quality"
        bad = (df[qcol].astype(str).str.startswith("unusable")
               if qcol in df.columns else pd.Series(False, index=df.index))
        out[p] = df[p].where(~bad)

    # AQI is already NaN where it was not computable.
    out["aqi"] = df["aqi"]
    out["aqi_category"] = df["aqi_category"]
    out["aqi_prominent_pollutant"] = df["aqi_prominent_pollutant"]
    out["aqi_n_pollutants"] = df["aqi_n_pollutants"]

    for w in WEATHER + ["wind_dir_resultant"]:
        if w in df.columns:
            out[w] = df[w]
    return out


# ----------------------------------------------------------------------------
def build_station(g, feats, dictionary):
    """All features for one station. `g` is sorted by date with a complete
    daily index, so shift(1) is genuinely 'yesterday'."""
    g = g.sort_values("Date")

    def record(name, group, window, leakage, note):
        dictionary.append(dict(feature=name, group=group, window=window,
                               leakage=leakage, note=note))

    # ---- core series: lags, rolling mean/sd, change rates -------------------
    for col in CORE:
        if col not in g.columns:
            continue
        past = g[col].shift(1)          # everything downstream starts at t-1

        for L in LAGS:
            name = f"{col}_lag{L}"
            feats[name] = g[col].shift(L)
            record(name, "lag", f"t-{L}", "safe",
                   "observed value L days before the target day")

        for W in ROLL_WINDOWS:
            r = past.rolling(W, min_periods=min_periods(W))
            feats[f"{col}_rmean{W}"] = r.mean()
            feats[f"{col}_rstd{W}"] = r.std()
            record(f"{col}_rmean{W}", "rolling", f"t-{W} to t-1", "safe",
                   f"mean of the {W} days before the target day")
            record(f"{col}_rstd{W}", "rolling", f"t-{W} to t-1", "safe",
                   f"variability over the {W} days before the target day")

        # Momentum: how fast it was already moving at the forecast origin.
        feats[f"{col}_chg1"] = g[col].shift(1) - g[col].shift(2)
        feats[f"{col}_chg7"] = g[col].shift(1) - g[col].shift(8)
        record(f"{col}_chg1", "change", "t-2 to t-1", "safe",
               "day-over-day change as of the forecast origin")
        record(f"{col}_chg7", "change", "t-8 to t-1", "safe",
               "week-over-week change as of the forecast origin")

    # ---- secondary pollutants: light context only --------------------------
    for col in SECONDARY:
        if col not in g.columns:
            continue
        feats[f"{col}_lag1"] = g[col].shift(1)
        feats[f"{col}_rmean7"] = g[col].shift(1).rolling(
            7, min_periods=min_periods(7)).mean()
        record(f"{col}_lag1", "lag", "t-1", "safe",
               "yesterday's level; weather adds little for this pollutant")
        record(f"{col}_rmean7", "rolling", "t-7 to t-1", "safe",
               "recent level of a pollutant that rarely sets the AQI")

    # ---- weather observed up to t-1 ----------------------------------------
    for w in WEATHER:
        if w not in g.columns:
            continue
        feats[f"wxobs_{w}_lag1"] = g[w].shift(1)
        record(f"wxobs_{w}_lag1", "weather-observed", "t-1", "safe",
               "yesterday's weather; known at forecast time")
        for W in WEATHER_ROLL:
            name = f"wxobs_{w}_rmean{W}"
            feats[name] = g[w].shift(1).rolling(
                W, min_periods=min_periods(W)).mean()
            record(name, "weather-observed", f"t-{W} to t-1", "safe",
                   f"{W}-day weather average ending at the forecast origin")

    # Cumulative rain: wet deposition has memory beyond a single day.
    if "rain_sum" in g.columns:
        for W in [3, 7]:
            name = f"wxobs_rain_cum{W}"
            feats[name] = g["rain_sum"].shift(1).rolling(
                W, min_periods=1).sum()
            record(name, "weather-observed", f"t-{W} to t-1", "safe",
                   f"total rainfall over the {W} days before the target day")

    # ---- weather on the target day -----------------------------------------
    if INCLUDE_SAME_DAY_WEATHER:
        for w in WEATHER:
            if w in g.columns:
                feats[f"wxfc_{w}"] = g[w]
                record(f"wxfc_{w}", "weather-same-day", "t",
                       "requires-forecast",
                       "weather on the target day; in production this is a "
                       "FORECAST, so evaluating on reanalysis overstates "
                       "accuracy")

        # Wind direction as sin/cos. Averaging bearings arithmetically is
        # wrong (350 deg and 10 deg average to 180, the opposite way), so the
        # resultant direction is decomposed instead.
        if "wind_dir_resultant" in g.columns:
            rad = np.deg2rad(g["wind_dir_resultant"])
            feats["wxfc_wind_dir_sin"] = np.sin(rad)
            feats["wxfc_wind_dir_cos"] = np.cos(rad)
            for nm in ["wxfc_wind_dir_sin", "wxfc_wind_dir_cos"]:
                record(nm, "weather-same-day", "t", "requires-forecast",
                       "cyclical encoding of wind direction")

        # Interactions, particulates only. NO2, SO2 and CO showed no weather
        # relationship above |rho| 0.12 after deseasonalising, so interaction
        # terms there would add dimensionality without signal.
        for p in ["PM2.5", "PM10"]:
            base = g[p].shift(1)        # yesterday's level, known at origin
            for w, why in [("wind_speed_mean", "dilution"),
                           ("temp_range", "nocturnal trapping"),
                           ("rain_sum", "wet deposition")]:
                if w not in g.columns:
                    continue
                name = f"ix_{p}_x_{w}"
                feats[name] = base * g[w]
                record(name, "interaction", "t-1 level x t weather",
                       "requires-forecast",
                       f"{why}; particulates only, where weather has signal")

    # ---- calendar ----------------------------------------------------------
    doy = g["Date"].dt.dayofyear
    feats["cal_doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    feats["cal_doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    dow = g["Date"].dt.dayofweek
    feats["cal_dow_sin"] = np.sin(2 * np.pi * dow / 7)
    feats["cal_dow_cos"] = np.cos(2 * np.pi * dow / 7)
    feats["cal_is_weekend"] = (dow >= 5).astype(int)
    for nm, note in [("cal_doy_sin", "annual cycle"), ("cal_doy_cos", "annual cycle"),
                     ("cal_dow_sin", "weekly traffic cycle"),
                     ("cal_dow_cos", "weekly traffic cycle"),
                     ("cal_is_weekend", "weekly traffic cycle")]:
        record(nm, "calendar", "t", "safe",
               f"{note}; the date is known in advance, so this is not leakage")

    return feats


def add_city_features(df, feats, dictionary):
    """Cross-station context: what the rest of the city was doing yesterday.
    Leave-one-out, so a station is never fed its own value back."""
    for col in ["aqi", "PM10"]:
        wide = df.pivot(index="Date", columns="Station", values=col)
        loo = pd.DataFrame({s: wide.drop(columns=s).mean(axis=1)
                            for s in wide.columns})
        stacked = loo.stack().rename("v").reset_index()
        stacked.columns = ["Date", "Station", "v"]
        stacked = stacked.sort_values(["Station", "Date"])
        stacked["v"] = stacked.groupby("Station")["v"].shift(1)

        key = df[["Date", "Station"]].merge(
            stacked, on=["Date", "Station"], how="left")
        name = f"city_{col}_lag1"
        feats[name] = key["v"].values
        dictionary.append(dict(
            feature=name, group="cross-station", window="t-1", leakage="safe",
            note="mean of the OTHER five stations yesterday; leave-one-out so "
                 "a station never sees its own value"))
    return feats


# ----------------------------------------------------------------------------
def leakage_audit(df, feats, n_samples=40, seed=0):
    """The real test of the leakage rule.

    For a sample of (station, date) rows, rebuild the features using ONLY the
    data up to and including that date, then confirm every `safe` feature on
    that row is unchanged. If any feature peeked at a future row, truncating
    the future changes its value and the check fails.

    `requires-forecast` features are excluded by construction: they legitimately
    use day t, which truncation keeps.
    """
    rng = np.random.default_rng(seed)
    stations = df["Station"].unique()
    dates = df["Date"].sort_values().unique()
    # Avoid the first 40 days, where windows are still warming up.
    candidates = dates[40:]

    safe_cols = [c for c, lk in zip(feats.columns, feats.attrs["leakage"])
                 if lk == "safe"]

    failures, checked = [], 0
    for _ in range(n_samples):
        st = rng.choice(stations)
        dt = pd.Timestamp(rng.choice(candidates))

        truncated = df[(df["Station"] == st) & (df["Date"] <= dt)].copy()
        if len(truncated) < 40:
            continue

        sub_feats = pd.DataFrame(index=truncated.index)
        build_station(truncated, sub_feats, [])

        full_row = feats.loc[
            (df["Station"] == st) & (df["Date"] == dt), safe_cols]
        trunc_row = sub_feats.loc[
            truncated.index[-1:], [c for c in safe_cols if c in sub_feats]]
        if full_row.empty or trunc_row.empty:
            continue

        checked += 1
        for c in trunc_row.columns:
            a = full_row.iloc[0][c]
            b = trunc_row.iloc[0][c]
            if pd.isna(a) and pd.isna(b):
                continue
            if pd.isna(a) != pd.isna(b) or not np.isclose(
                    float(a), float(b), rtol=1e-9, atol=1e-9, equal_nan=True):
                failures.append((st, str(dt.date()), c, a, b))

    return checked, failures


# ----------------------------------------------------------------------------
def main():
    df = pd.read_csv(IN_CSV, parse_dates=["Date"])
    print(f"Read {IN_CSV}  {df.shape}")

    base = masked_frame(df).sort_values(["Station", "Date"]).reset_index(drop=True)

    dictionary = []
    feats = pd.DataFrame(index=base.index)
    parts = []
    for st, g in base.groupby("Station", sort=False):
        f = pd.DataFrame(index=g.index)
        build_station(g, f, dictionary if not parts else [])
        parts.append(f)
    feats = pd.concat(parts).sort_index()

    feats = add_city_features(base, feats, dictionary)

    fd = pd.DataFrame(dictionary).drop_duplicates(subset="feature")
    fd = (fd.set_index("feature").reindex(feats.columns)
            .rename_axis("feature").reset_index())
    feats.attrs["leakage"] = fd["leakage"].fillna("safe").tolist()

    out = pd.concat([
        base[["Station", "Date"]],
        feats,
        base[[TARGET, "aqi_category", "aqi_n_pollutants"]],
    ], axis=1)

    # ---- validation --------------------------------------------------------
    print("\n" + "=" * 70)
    print("LEAKAGE AUDIT")
    print("=" * 70)
    print("Rebuilding features for sampled rows using only data up to that")
    print("date, then checking every 'safe' feature is unchanged.\n")
    checked, failures = leakage_audit(base, feats)
    print(f"  rows checked: {checked}")
    if failures:
        print(f"  [FAIL] {len(failures)} feature values changed when the")
        print("         future was truncated — those features look ahead:")
        for f in failures[:10]:
            print("   ", f)
        raise SystemExit(1)
    print("  [PASS] no 'safe' feature changed. None of them look ahead.")

    n_fc = int((fd["leakage"] == "requires-forecast").sum())
    print(f"\n  {n_fc} features are marked 'requires-forecast' — they use day-t")
    print("  weather, which is a FORECAST in production, not an observation.")
    print("  They are not leakage, but training and evaluating them on")
    print("  reanalysis overstates real-world accuracy.")

    print("\n" + "=" * 70)
    print("FEATURE SUMMARY")
    print("=" * 70)
    print(f"Total features: {len(feats.columns)}")
    print("\nBy group:")
    print(fd.groupby("group").size().to_string())
    print("\nBy leakage class:")
    print(fd.groupby("leakage").size().to_string())

    print("\nCoverage (rows with a usable value), worst 10 features:")
    cov = (feats.notna().mean() * 100).round(1).sort_values()
    print(cov.head(10).to_string())
    print("\n  Low coverage is expected for long windows at the start of each")
    print("  station's series, and wherever masked cells thin a window below")
    print(f"  the {MIN_WINDOW_FRAC:.0%} minimum.")

    usable_rows = out[TARGET].notna() & feats.notna().all(axis=1)
    print(f"\nRows with a target and every feature present: "
          f"{int(usable_rows.sum()):,} of {len(out):,}")
    print("  Tree models handle NaN natively, so rows with some missing")
    print("  features are still usable. This figure is the strict case.")

    out.to_csv(OUT_CSV, index=False)
    fd.to_csv(OUT_DICT, index=False)
    print(f"\nWrote {OUT_CSV}  {out.shape}")
    print(f"Wrote {OUT_DICT}  ({len(fd)} features documented)")

    print("\nBefore modelling: split chronologically, never randomly. A random")
    print("split puts tomorrow in training and yesterday in test, and every")
    print("lag feature then carries test-period information.")


if __name__ == "__main__":
    main()

Read Bengaluru_AQI_daily.csv  (6576, 79)

LEAKAGE AUDIT
Rebuilding features for sampled rows using only data up to that
date, then checking every 'safe' feature is unchanged.

  rows checked: 40
  [PASS] no 'safe' feature changed. None of them look ahead.

  14 features are marked 'requires-forecast' — they use day-t
  weather, which is a FORECAST in production, not an observation.
  They are not leakage, but training and evaluating them on
  reanalysis overstates real-world accuracy.

FEATURE SUMMARY
Total features: 94

By group:
group
calendar             5
change               6
cross-station        2
interaction          6
lag                 19
rolling             28
weather-observed    20
weather-same-day     8

By leakage class:
leakage
requires-forecast    14
safe                 80

Coverage (rows with a usable value), worst 10 features:
CO_rmean7        77.5
CO_lag1          77.8
NO2_rmean7       81.0
NO2_lag1         81.9
Ozone_rmean7     93.0
Ozone_lag1       93.6
PM2.5_rst